# 模型 artifact 与矩阵 manifest v1

目标：验证 `src/model` 不只训练预测，还能保存/加载模型 artifact，并让模型矩阵输出统一 `artifact_manifest.json`。

边界：本轮仍使用 smoke adapter，验证的是稳定持久化契约，不是论文严格模型复现。

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact, load_model_artifact, save_model_artifact
from src.model.matrix import run_factor_feature_set_matrix

CARD = Path('research/model_artifact_manifest_v1')
CARD.mkdir(parents=True, exist_ok=True)
price = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(price, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
test = dataset.frame.loc[dataset.frame['date'] > '2026-04-30'].copy()
feature_columns = dataset.feature_columns[:5]
len(train), len(test), feature_columns

(82,
 26,
 ('soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d'))

In [2]:
artifact = fit_model_artifact('ridge_mse', train=train, feature_columns=feature_columns)
before = artifact.predict(test)
artifact_path = save_model_artifact(CARD / 'ridge_mse_artifact.joblib', artifact)
loaded = load_model_artifact(artifact_path)
after = loaded.predict(test)
before.to_csv(CARD / 'artifact_predictions_before.csv', index=False)
after.to_csv(CARD / 'artifact_predictions_after.csv', index=False)
{
    'artifact_path': str(artifact_path),
    'metadata': loaded.metadata_payload(),
    'predictions_match_after_load': bool(before['prediction'].equals(after['prediction'])),
}

{'artifact_path': 'research/model_artifact_manifest_v1/ridge_mse_artifact.joblib',
 'metadata': {'schema_version': 'model_artifact.v1',
  'model_name': 'ridge_mse',
  'feature_columns': ['soxl_return_1d',
   'soxx_return_1d',
   'soxl_return_5d',
   'soxx_return_5d',
   'soxl_soxx_relative_5d'],
  'objective': 'mse',
  'family': 'linear',
  'adapter_status': 'smoke_adapter',
  'target_column': 'target_return',
  'train_rows': 82},
 'predictions_match_after_load': True}

In [3]:
manifest_outputs = run_factor_feature_set_matrix(
    card_dir=CARD / 'matrix_card',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse',),
    top_n_factors=3,
)
manifest = json.loads((CARD / 'matrix_card' / 'artifact_manifest.json').read_text(encoding='utf-8'))
{
    'manifest_hash': manifest['manifest_hash'],
    'artifact_type': manifest['artifact_type'],
    'dataset': manifest['dataset'],
    'paths': manifest['paths'],
    'metadata': manifest['metadata'],
}

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'manifest_hash': '8cce95aa6446cd0ce40ecfa65316e29035f83a399486c728d3206d8709796b52',
 'artifact_type': 'model_matrix',
 'dataset': {'calendar_path': 'data/qlib/soxl/calendars/day.txt',
  'data_paths': ['data/processed/panels/soxl_soxx_vix_full.csv'],
  'end': '2026-06-09',
  'metadata': {'source': 'load_soxl_support_frame'},
  'name': 'soxl_support_daily',
  'schema_version': 'dataset.v1',
  'start': '2026-01-01',
  'universe': ['SOXL', 'SOXX', 'VIX']},
 'paths': {'analysis_summary': 'analysis_summary.json',
  'daily_backtest': 'daily_backtest.csv',
  'factor_ic': 'factor_ic.csv',
  'feature_set_specs': 'feature_set_specs.json',
  'model_comparison': 'model_comparison.csv',
  'model_effect_deltas': 'model_effect_deltas.csv',
  'predictions': 'predictions.csv',
  'qlib_risk_metrics': 'qlib_risk_metrics.csv',
  'regime_metrics': 'regime_metrics.csv',
  'window_metrics': 'window_metrics.csv'},
 'metadata': {'adapter_status': 'smoke_adapter',
  'base_model_count': 1,
  'feature_set_count'

In [4]:
summary = {
    'schema_version': 'model_artifact_manifest_review.v1',
    'model_artifact': str(artifact_path),
    'model_name': loaded.model_name,
    'feature_columns': list(loaded.feature_columns),
    'prediction_rows': int(len(after)),
    'predictions_match_after_load': bool(before['prediction'].equals(after['prediction'])),
    'matrix_manifest': 'matrix_card/artifact_manifest.json',
    'manifest_hash': manifest['manifest_hash'],
    'manifest_path_count': len(manifest['paths']),
    'matrix_status': manifest['metadata']['status'],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'model_artifact_manifest_review.v1',
 'model_artifact': 'research/model_artifact_manifest_v1/ridge_mse_artifact.joblib',
 'model_name': 'ridge_mse',
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d'],
 'prediction_rows': 26,
 'predictions_match_after_load': True,
 'matrix_manifest': 'matrix_card/artifact_manifest.json',
 'manifest_hash': '8cce95aa6446cd0ce40ecfa65316e29035f83a399486c728d3206d8709796b52',
 'manifest_path_count': 10,
 'matrix_status': 'feature_set_matrix_completed'}

## 中文结论

- 做了什么：训练 `ridge_mse` smoke artifact，保存为 joblib 后重新加载，并复现预测；同时运行 feature-set 矩阵并写出 `artifact_manifest.json`。
- 核心指标：prediction rows=26，加载后预测一致=`True`，manifest hash=`8cce95aa6446cd0ce40ecfa65316e29035f83a399486c728d3206d8709796b52`。
- regime 差异：矩阵 sidecar 仍输出 `regime_metrics.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`，并在 manifest 中登记。
- 是否晋升：否。本轮是模型持久化和 manifest contract 验证，不代表模型进入 production。
- 风险：当前保存的是 smoke adapter estimator；论文严格实现替换时应保持同一 artifact schema 和 manifest 字段。
- 下一步：逐步替换高优先级模型的 smoke adapter，并把每个严格实现的 artifact 写入相同 manifest 链路。